run clickhouse dataloaders in any order. there are 4. the 100k load can be a burden on a system, and timeouts may occur. use the markdown cell to truncate the table if needed.


In [1]:
import clickhouse_connect

from clickhouse_connect.driver.tools import insert_file
import pyarrow
import pandas as pd


In [2]:
client = clickhouse_connect.get_client(host='localhost', username='default', password='', send_receive_timeout=1300)

In [11]:
client.command("""
create table IF NOT EXISTS idnet100k_no_dupe
(
    name             text,
    imageData      text,
    prelabeled_fraud integer,
    id               text,
    person_name      text,
    address          text,
    birthday         text,
    gender           text,
    ethnicity        Nullable(text),
    class            text,
    issue_date       text,
    expire_date      text,
    eye_color        text,
    hair_color       Nullable(text),
    is_donor         text,
    is_veteran       text,
    license_number   text,
    height_inches    integer,
    weight_lbs       integer
)
    ENGINE = MergeTree
ORDER BY name
SETTINGS allow_nullable_key = 1
""")

In [3]:
df = pd.read_parquet('../silver/idnet100k_no_dupe.parquet')

In [14]:
table_name = "idnet100k_no_dupe"
chunk_size = 1000  # Adjust based on memory and row width

#if an error occurs during load, clickhouse may not let you truncate. unhide and run this if you need to
client.command("truncate table idnet100k_no_dupe SETTINGS max_table_size_to_drop = 0")

client.command("select count(*) from idnet100k_no_dupe")

In [15]:
import time

total_rows = len(df)
print(f"Starting batch insert of {total_rows:,} rows...")

for start_idx in range(0, total_rows, chunk_size):
    # Slice the dataframe into a smaller chunk
    end_idx = min(start_idx + chunk_size, total_rows)
    df_chunk = df.iloc[start_idx:end_idx]

    #client = clickhouse_connect.get_client(host='localhost', username='default', password='', send_receive_timeout=1300)

    # Insert the chunk
    client.insert_df(table=table_name, df=df_chunk, settings={
        "send_timeout": 1300,            # Max time server spends sending data
        "receive_timeout": 1300,         # Max time server spends receiving data from client
        "max_execution_time": 1300,      # Prevents ClickHouse from aborting long-running queries
        "timeout_before_checking_execution_speed": 1300,  # Stops aborts from low-speed thresholds
        "connect_timeout": 1300, })

    #client.close()

    time.sleep(60)
    print(f"Successfully inserted rows {start_idx:,} to {end_idx:,}")

Starting batch insert of 106,060 rows...
Successfully inserted rows 0 to 1,000
Successfully inserted rows 1,000 to 2,000
Successfully inserted rows 2,000 to 3,000
Successfully inserted rows 3,000 to 4,000
Successfully inserted rows 4,000 to 5,000
Successfully inserted rows 5,000 to 6,000
Successfully inserted rows 6,000 to 7,000
Successfully inserted rows 7,000 to 8,000
Successfully inserted rows 8,000 to 9,000
Successfully inserted rows 9,000 to 10,000
Successfully inserted rows 10,000 to 11,000
Successfully inserted rows 11,000 to 12,000
Successfully inserted rows 12,000 to 13,000
Successfully inserted rows 13,000 to 14,000
Successfully inserted rows 14,000 to 15,000
Successfully inserted rows 15,000 to 16,000
Successfully inserted rows 16,000 to 17,000
Successfully inserted rows 17,000 to 18,000
Successfully inserted rows 18,000 to 19,000
Successfully inserted rows 19,000 to 20,000
Successfully inserted rows 20,000 to 21,000
Successfully inserted rows 21,000 to 22,000
Successfully i

In [8]:
client.close()